**Experimentation - Contextual Bandits**

In some cases there may be no relevant historical data available. To decide the optimal treatment, we can do experiment. 

In [ ]:
import pandas as pd
import numpy as np
from random import choice, random
from causalml.inference.meta import BaseTClassifier, BaseSClassifier, BaseXClassifier
from causalml.inference.tree import UpliftRandomForestClassifier
from xgboost import XGBClassifier, XGBRegressor
import plotly.express as px
import plotly.io as pio

# Embed plotly.js in the cell output so the interactive chart survives
# an HTML export via nbconvert.
pio.renderers.default = "notebook"

In [2]:
df_full = pd.read_csv("Mediamill_data.csv")
df_full = df_full.loc[0:19999,]

In [3]:
df_full.head()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,y1,y2,y3,y4,y5,y6,y7,y8,y9,y10
0,0.38088,0.49408,0.54001,0.42293,0.15832,0.32698,0.39086,0.52712,0.25405,0.22373,1,0,0,1,0,1,0,0,0,0
1,0.44957,0.46049,0.45347,0.41078,0.23176,0.40215,0.34959,0.53646,0.31812,0.30162,1,1,0,0,0,1,0,0,0,0
2,0.41680,0.54900,0.52085,0.46541,0.18160,0.35725,0.38935,0.53019,0.29094,0.24452,1,1,0,0,0,0,0,0,0,0
3,0.50199,0.48082,0.43554,0.43200,0.25060,0.40835,0.35782,0.49919,0.35317,0.32762,1,1,0,0,0,1,0,0,0,0
4,0.50682,0.48756,0.44496,0.41937,0.24502,0.40251,0.36114,0.49052,0.35760,0.32960,1,1,0,1,0,1,1,0,1,0


In [4]:
n = df_full.shape[0]
p_k = pd.Series([i[0] for i in df_full.columns]).value_counts()
p = p_k["x"]
k = p_k["y"]

print("n:", n)
print("p:", p)
print("k:", k)

n: 20000
p: 10
k: 10


In [5]:
first_batch_size = k * 100
fit_batch_size = 500
nbatches = int(np.ceil((n - first_batch_size) / fit_batch_size))

print(first_batch_size)
print(nbatches)

1000
38


In [6]:
y_names = df_full.columns[df_full.columns.str.startswith("y")] 
x_names = df_full.columns[df_full.columns.str.startswith("x")] 
treatment_list = [i.replace("y", "treatment") for i in y_names]
treat_nms = treatment_list[1:]

print(y_names)
print(x_names)
print(treatment_list)
print(treat_nms)

Index(['y1', 'y2', 'y3', 'y4', 'y5', 'y6', 'y7', 'y8', 'y9', 'y10'], dtype='object')
Index(['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10'], dtype='object')
['treatment1', 'treatment2', 'treatment3', 'treatment4', 'treatment5', 'treatment6', 'treatment7', 'treatment8', 'treatment9', 'treatment10']
['treatment2', 'treatment3', 'treatment4', 'treatment5', 'treatment6', 'treatment7', 'treatment8', 'treatment9', 'treatment10']


In [7]:
df_full[y_names].mean()

y1     0.77150
y2     0.63855
y3     0.36295
y4     0.32535
y5     0.19935
y6     0.19485
y7     0.12655
y8     0.13955
y9     0.10535
y10    0.10415
dtype: float64

In [8]:
df_first_batch = df_full.copy().loc[0 : (first_batch_size - 1),] 

print(df_first_batch.head())
print(df_first_batch.shape)

        x1       x2       x3       x4       x5       x6       x7       x8  \
0  0.38088  0.49408  0.54001  0.42293  0.15832  0.32698  0.39086  0.52712   
1  0.44957  0.46049  0.45347  0.41078  0.23176  0.40215  0.34959  0.53646   
2  0.41680  0.54900  0.52085  0.46541  0.18160  0.35725  0.38935  0.53019   
3  0.50199  0.48082  0.43554  0.43200  0.25060  0.40835  0.35782  0.49919   
4  0.50682  0.48756  0.44496  0.41937  0.24502  0.40251  0.36114  0.49052   

        x9      x10  y1  y2  y3  y4  y5  y6  y7  y8  y9  y10  
0  0.25405  0.22373   1   0   0   1   0   1   0   0   0    0  
1  0.31812  0.30162   1   1   0   0   0   1   0   0   0    0  
2  0.29094  0.24452   1   1   0   0   0   0   0   0   0    0  
3  0.35317  0.32762   1   1   0   0   0   1   0   0   0    0  
4  0.35760  0.32960   1   1   0   1   0   1   1   0   1    0  
(1000, 20)


In [9]:
first_batch_treatments = [choice(treatment_list) for i in range(first_batch_size)]
first_batch_treatments_index = [int(first_batch_treatments[i][9:]) - 1 for i in range(first_batch_size)]
y_obs_first_batch = df_first_batch[y_names].to_numpy()[np.arange(first_batch_size), first_batch_treatments_index]

df = pd.concat([df_first_batch[x_names], pd.DataFrame(first_batch_treatments), pd.DataFrame(y_obs_first_batch)], axis=1)
df.columns = df.columns.tolist()[:-2] + ["treatment", "target"]
names = df.columns.tolist()

In [10]:
print(df.head())
print(df.shape)

        x1       x2       x3       x4       x5       x6       x7       x8  \
0  0.38088  0.49408  0.54001  0.42293  0.15832  0.32698  0.39086  0.52712   
1  0.44957  0.46049  0.45347  0.41078  0.23176  0.40215  0.34959  0.53646   
2  0.41680  0.54900  0.52085  0.46541  0.18160  0.35725  0.38935  0.53019   
3  0.50199  0.48082  0.43554  0.43200  0.25060  0.40835  0.35782  0.49919   
4  0.50682  0.48756  0.44496  0.41937  0.24502  0.40251  0.36114  0.49052   

        x9      x10    treatment  target  
0  0.25405  0.22373   treatment8       0  
1  0.31812  0.30162   treatment2       1  
2  0.29094  0.24452  treatment10       0  
3  0.35317  0.32762   treatment9       0  
4  0.35760  0.32960  treatment10       0  
(1000, 12)


In [11]:
control_name = "treatment1"
causal_model = ["S-Learner", "T-Learner", "X-Learner"][1]
if causal_model == "T-Learner":
    init_model = BaseTClassifier(XGBClassifier(), control_name = control_name)
elif causal_model == "X-Learner":
    init_model = BaseXClassifier(outcome_learner=XGBClassifier(), effect_learner=XGBRegressor(), control_name=control_name)
else:
    init_model = BaseSClassifier(XGBClassifier(), control_name = control_name)

In [12]:
def recommend(cate, control_name, method="greedy", epsilon=0.05):
    if method == "greedy":
        out = np.where((cate < 0).all(axis=1), control_name, cate.idxmax(axis=1))
    elif method == "epsilon_greedy":
        out = np.where((cate < 0).all(axis=1), control_name, cate.idxmax(axis=1))
        for i in range(len(cate)):
            if random() < epsilon:
                out[i] = choice(treatment_list)
    return out

In [13]:
df_gr = df.copy()
df_egr = df.copy()

decay = 0.8
epsilon = 0.05

for i in range(nbatches):
    
    print("Batch: ", i)
    print("epsilon:", epsilon)
    
    llim = first_batch_size + fit_batch_size * i
    ulim = first_batch_size + fit_batch_size * (i + 1) - 1
    if ulim > (n - 1):
        ulim = n - 1
    
    df_full_i = df_full.loc[llim:ulim, :]
    df_full_i.index = range(len(df_full_i))
    
    #
    init_model.fit(X=df_gr[x_names].values, treatment=df_gr["treatment"].values, y=df_gr["target"].values)
    if causal_model in ["T-Learner", "S-Learner", "X-Learner"]:
        pred_gr_i = init_model.predict(df_full_i[x_names])
        pred_gr_i = pd.DataFrame(pred_gr_i, columns = treat_nms)
    # elif causal_model == "CRF":
    #     pred_gr_i_full = init_model.predict(df_full_i[x_names], full_output = True)
    #     cols_sel = pred_gr_i_full.columns[pred_gr_i_full.columns.str.startswith("delta")]
    #     pred_gr_i = pred_gr_i_full[cols_sel]
    #     pred_gr_i.columns = [i[6:] for i in pred_gr_i.columns]
    reco_gr_i = recommend(pred_gr_i.copy(), control_name, "greedy")

    df_gr_i = pd.concat([df_full_i[x_names], pd.Series(reco_gr_i), 
                         df_full_i[y_names].iloc[:, int(reco_gr_i[0][-1])]], 
                        axis=1)
    df_gr_i.columns = names
    df_gr = pd.concat([df_gr, df_gr_i], axis=0)
    
    #
    init_model.fit(X=df_egr[x_names].values, treatment=df_egr["treatment"].values, y=df_egr["target"].values)
    if causal_model in ["T-Learner", "S-Learner"]:
        pred_egr_i = init_model.predict(df_full_i[x_names])
        pred_egr_i = pd.DataFrame(pred_egr_i, columns = treat_nms)
    # elif causal_model == "CRF":
    #     pred_egr_i_full = init_model.predict(df_full_i[x_names], full_output = True)
    #     cols_sel = pred_egr_i_full.columns[pred_egr_i_full.columns.str.startswith("delta")]
    #     pred_egr_i = pred_egr_i_full[cols_sel]
    #     pred_egr_i.columns = [i[6:] for i in pred_egr_i.columns]
    reco_egr_i = recommend(pred_egr_i.copy(), control_name, "epsilon_greedy", epsilon)

    epsilon = epsilon * decay
    
    df_egr_i = pd.concat([df_full_i[x_names], pd.Series(reco_egr_i), 
                         df_full_i[y_names].iloc[:, int(reco_egr_i[0][-1])]], 
                        axis=1)
    df_egr_i.columns = names
    df_egr = pd.concat([df_egr, df_egr_i], axis=0)

df_gr.reset_index(drop=True, inplace=True)
df_egr.reset_index(drop=True, inplace=True)

Batch:  0
epsilon: 0.05
Batch:  1
epsilon: 0.04000000000000001
Batch:  2
epsilon: 0.03200000000000001
Batch:  3
epsilon: 0.025600000000000008
Batch:  4
epsilon: 0.02048000000000001
Batch:  5
epsilon: 0.016384000000000006
Batch:  6
epsilon: 0.013107200000000006
Batch:  7
epsilon: 0.010485760000000005
Batch:  8
epsilon: 0.008388608000000004
Batch:  9
epsilon: 0.006710886400000004
Batch:  10
epsilon: 0.005368709120000003
Batch:  11
epsilon: 0.0042949672960000025
Batch:  12
epsilon: 0.0034359738368000023
Batch:  13
epsilon: 0.002748779069440002
Batch:  14
epsilon: 0.002199023255552002
Batch:  15
epsilon: 0.0017592186044416017
Batch:  16
epsilon: 0.0014073748835532814
Batch:  17
epsilon: 0.0011258999068426252
Batch:  18
epsilon: 0.0009007199254741002
Batch:  19
epsilon: 0.0007205759403792802
Batch:  20
epsilon: 0.0005764607523034242
Batch:  21
epsilon: 0.00046116860184273935
Batch:  22
epsilon: 0.0003689348814741915
Batch:  23
epsilon: 0.00029514790517935324
Batch:  24
epsilon: 0.0002361183

In [14]:
# random treatment allocation policy
n_rest = n-len(df)
df_rest = df_full.copy().loc[first_batch_size:,]
df_rest.reset_index(drop=True, inplace=True)

rand_treat = [choice(treatment_list) for i in range(n_rest)]
rand_treat_index = [int(rand_treat[i][9:]) - 1 for i in range(n_rest)]
rand_y = df_rest[y_names].to_numpy()[np.arange(n_rest), rand_treat_index]
df_rand = pd.concat([df_rest[x_names], pd.DataFrame(rand_treat), pd.DataFrame(rand_y)], axis=1)
df_rand.columns = x_names.tolist() + ["treatment", "target"]
df_rand = pd.concat([df, df_rand], axis=0)
df_rand.reset_index(drop=True, inplace=True)

In [15]:
print("Greedy allocation")
print(df_gr.treatment.value_counts())
print("---")
print("Epsilon-Greedy allocation")
print(df_egr.treatment.value_counts())
print("---")
print("Random allocation")
df_rand.treatment.value_counts()

Greedy allocation
treatment
treatment3     11772
treatment7      1804
treatment5      1704
treatment1      1279
treatment6      1038
treatment9       696
treatment8       572
treatment4       545
treatment2       311
treatment10      279
Name: count, dtype: int64
---
Epsilon-Greedy allocation
treatment
treatment3     11251
treatment7      1587
treatment5      1425
treatment1      1355
treatment6      1022
treatment4       920
treatment9       814
treatment8       797
treatment2       430
treatment10      399
Name: count, dtype: int64
---
Random allocation


treatment
treatment2     2067
treatment7     2063
treatment3     2043
treatment9     2005
treatment6     1997
treatment10    1983
treatment5     1978
treatment8     1961
treatment1     1953
treatment4     1950
Name: count, dtype: int64

In [16]:
df_gr.head()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,treatment,target
0,0.38088,0.49408,0.54001,0.42293,0.15832,0.32698,0.39086,0.52712,0.25405,0.22373,treatment8,0
1,0.44957,0.46049,0.45347,0.41078,0.23176,0.40215,0.34959,0.53646,0.31812,0.30162,treatment2,1
2,0.41680,0.54900,0.52085,0.46541,0.18160,0.35725,0.38935,0.53019,0.29094,0.24452,treatment10,0
3,0.50199,0.48082,0.43554,0.43200,0.25060,0.40835,0.35782,0.49919,0.35317,0.32762,treatment9,0
4,0.50682,0.48756,0.44496,0.41937,0.24502,0.40251,0.36114,0.49052,0.35760,0.32960,treatment10,0


In [17]:
df_egr.head()

,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,treatment,target
0,0.38088,0.49408,0.54001,0.42293,0.15832,0.32698,0.39086,0.52712,0.25405,0.22373,treatment8,0
1,0.44957,0.46049,0.45347,0.41078,0.23176,0.40215,0.34959,0.53646,0.31812,0.30162,treatment2,1
2,0.41680,0.54900,0.52085,0.46541,0.18160,0.35725,0.38935,0.53019,0.29094,0.24452,treatment10,0
3,0.50199,0.48082,0.43554,0.43200,0.25060,0.40835,0.35782,0.49919,0.35317,0.32762,treatment9,0
4,0.50682,0.48756,0.44496,0.41937,0.24502,0.40251,0.36114,0.49052,0.35760,0.32960,treatment10,0


In [18]:
divide_reward = list(range(1, (n + 1)))

av_reward_gr = df_gr.target.cumsum() / divide_reward
av_reward_egr = df_egr.target.cumsum() / divide_reward
av_reward_rand = df_rand.target.cumsum() / divide_reward

df_plot = pd.concat([av_reward_gr, av_reward_egr, av_reward_rand], axis=1)
df_plot.columns = ["Greedy", "Epsilon-Greedy", "Random"]
df_plot["round"] = range(len(df_plot))

print(df_plot.head())
print(df_plot.tail())


     Greedy  Epsilon-Greedy    Random  round
0  0.000000        0.000000  0.000000      0
1  0.500000        0.500000  0.500000      1
2  0.333333        0.333333  0.333333      2
3  0.250000        0.250000  0.250000      3
4  0.200000        0.200000  0.200000      4
         Greedy  Epsilon-Greedy    Random  round
19995  0.283107        0.305711  0.301060  19995
19996  0.283092        0.305696  0.301045  19996
19997  0.283128        0.305731  0.301030  19997
19998  0.283114        0.305715  0.301015  19998
19999  0.283100        0.305700  0.301050  19999


In [19]:
plt = px.line(df_plot, x = "round", y = df_plot.columns)
plt

- Change epsilon
- Change first batch size
- Change batch sizes
- Change the causal model
- Change decay rate
- Change the sample size and who are in the sample